In [1]:
import time
import cv2
import numpy as np
import os
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from notebooks.resident4 import Re4CNN, RE4Env
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from pathlib import Path
from stable_baselines3.common.policies import ActorCriticPolicy
from sdlarch_rl.utils.utils import get_last_index
import pygame
import torch as th

train_path="./imitation/"
sub_train_path = train_path + "steps"
DETERMINISTIC=False

last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")

global myEnv

def make_env():
    def _init():
        global myEnv
        myEnv = RE4Env()

        env = WarpFrame(myEnv, width=128, height=128)
        
        return env
    return _init

env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4)

SCALE = 3
SCREEN_WIDTH = 640 * SCALE
SCREEN_HEIGHT = 480 * SCALE

prev_keys = set()

env.reset()

could_use_ai = False

color=(0, 0, 255)

current_epoch = 0

model = None

pygame.init()

window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

NUMBER_OF_ACTIONS=18 

COUNT_FRAME = 0
last_action = None

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


while True:
    pygame.event.pump()

    heatmap_resized = None

    action = [np.zeros(NUMBER_OF_ACTIONS, dtype=np.int8)]

    text_to_display = "Current epoch: " + str(current_epoch)

    if keyboard.is_pressed('k'):
        could_use_ai = not could_use_ai

        if could_use_ai:
            color=(0, 255, 0)
            current_epoch += 1

            COUNT_FRAME = 0

            if current_epoch > last_index_imitation:
                current_epoch = last_index_imitation

            # TODO: comment
            current_epoch = 14 # 73 # 75 # 45 # 35

            latest_model_path = sub_train_path + f"/bc_policy{current_epoch}.zip"

            print("loading from: " + str(latest_model_path))
            
            model = ActorCriticPolicy.load(
                str(latest_model_path),
            )

            total, trainable = count_params(model)
    
            print(f"Total params: {total:,}")
            print(f"Trainable params: {trainable:,}")
        else:
            color=(0, 0, 255)
        
        time.sleep(0.5)

        print("k is pressed: ", could_use_ai)

    if not could_use_ai:
        model = None
        # dpad
        if keyboard.is_pressed('up'):
            action[0][0] = 1
        if keyboard.is_pressed('down'):
            action[0][1] = 1
        if keyboard.is_pressed('left'):
            action[0][2] = 1
        if keyboard.is_pressed('right'):
            action[0][3] = 1
    
        # buttons
        if keyboard.is_pressed('i'):
            action[0][4] = 1
        if keyboard.is_pressed('o'):
            action[0][5] = 1
        if keyboard.is_pressed('p'):
            action[0][6] = 1
    else:
        if model is not None:
            if COUNT_FRAME % 4 == 0 and COUNT_FRAME > 0:
                action, _ = model.predict(obs, deterministic=DETERMINISTIC)
                last_action = action

            COUNT_FRAME = COUNT_FRAME + 1
            action = last_action

    if last_action is None:
        last_action = action

    obs, _, _, _ = env.step(action)

    img = myEnv.render()

    if img is not None:

    
        # img = obs[0, :, :, 0]
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        # img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        # img = cv2.resize(img, (128, 128))
        img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
        
        cv2.circle(img, center=(100, 100), radius=50, color=color, thickness=2)
        
        text_to_display = "Current Epoch: " + str(current_epoch)
        position = (50, 250)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.5
        thickness = 2
        line_type = cv2.LINE_AA


        cv2.putText(img, text_to_display, position, font, font_scale, color, thickness, line_type)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        surface = pygame.image.frombuffer(
            img,
            (img.shape[1], img.shape[0]),
            "RGB"
        )

        window.blit(surface, (0, 0))
        
        
    pygame.display.flip()

pygame.quit()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


None
next search...
next search...
next search...
next search...
next search...
next search...
Window founded HWND: 1968144.
loading from: ./imitation/steps/bc_policy14.zip


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)


Total params: 11,152,692
Trainable params: 11,152,692
k is pressed:  True
k is pressed:  False
loading from: ./imitation/steps/bc_policy14.zip
Total params: 11,152,692
Trainable params: 11,152,692
k is pressed:  True


error: (1400, 'GetClientRect', 'O identificador da janela é inválido.')